# Model Results Comparison

This notebook loads saved model evaluation results and delegates shared comparison plots to `src.evaluation.visualizer`, so the notebook stays aligned with the pipeline figures. It expects `model_results.csv`, usually produced by `main.py` under `output/results/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.visualizer import (
    plot_feature_impact,
    plot_rmse_r2_bars,
    plot_summary_heatmap,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RESULT_CANDIDATES = [
    ROOT / "output" / "results" / "model_results.csv",
    ROOT / "notebooks" / "output" / "results" / "model_results.csv",
]
PLOTS_DIR = ROOT / "output" / "plots"

RESULT_PATH = next((path for path in RESULT_CANDIDATES if path.exists()), None)
RESULT_PATH


In [ ]:
if RESULT_PATH is None:
    searched = "\n".join(f"- {path}" for path in RESULT_CANDIDATES)
    raise FileNotFoundError(
        "Could not find model_results.csv. Run the training pipeline first, for example:\n"
        "    python main.py --skip-deep\n\n"
        f"Searched:\n{searched}"
    )

results = pd.read_csv(RESULT_PATH)
required_columns = {"model", "features", "cv_rmse", "test_rmse", "cv_r2", "test_r2"}
missing_columns = sorted(required_columns - set(results.columns))
if missing_columns:
    raise ValueError(f"model_results.csv is missing required columns: {missing_columns}")

results


## Best Models

Lower RMSE, MAE, and MARD are better. Higher R2 is better.


In [ ]:
metric_cols = [
    col
    for col in ["test_rmse", "test_mae", "test_mard", "test_r2", "cv_rmse", "cv_r2"]
    if col in results.columns
]
ranked = results.sort_values("test_rmse").reset_index(drop=True)
ranked[["model", "features", *metric_cols]].style.format(precision=4)


## Shared Comparison Figures

These cells call the same visualizer functions used by `main.py`.


In [ ]:
plot_rmse_r2_bars(results)


In [ ]:
plot_feature_impact(results)


In [ ]:
plot_summary_heatmap(results)


## Cross-Validation vs Test Performance

This notebook-only diagnostic compares cross-validation and test RMSE.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=results,
    x="cv_rmse",
    y="test_rmse",
    hue="model",
    style="features",
    s=120,
    ax=ax,
)
lo = min(results["cv_rmse"].min(), results["test_rmse"].min())
hi = max(results["cv_rmse"].max(), results["test_rmse"].max())
ax.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1)
ax.set_title("CV RMSE vs Test RMSE")
ax.set_xlabel("CV RMSE")
ax.set_ylabel("Test RMSE")
plt.tight_layout()


## Save Figures

Use this optional cell to save the reusable comparison figures as PNG files.


In [ ]:
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

saved_paths = {
    "rmse_r2_bars": PLOTS_DIR / "rmse_r2_bars.png",
    "feature_impact": PLOTS_DIR / "feature_impact.png",
    "summary_heatmap": PLOTS_DIR / "summary_heatmap.png",
    "model_results_comparison": PLOTS_DIR / "model_results_comparison.png",
}

plot_rmse_r2_bars(results, save_path=saved_paths["rmse_r2_bars"])
plot_feature_impact(results, save_path=saved_paths["feature_impact"])
plot_summary_heatmap(results, save_path=saved_paths["summary_heatmap"])
plot_summary_heatmap(results, save_path=saved_paths["model_results_comparison"])

saved_paths
